# saveとload

## early stopping

In [1]:
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
import torch
from torch import nn
from torch.nn import functional as F
from torch import optim
from torch.utils.data import Dataset
from sklearn import datasets
from sklearn.model_selection import train_test_split

In [ ]:
class MyDataset(Dataset):
    def __init__(self, X, y, transform = None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        X = self.X[idx]
        y = self.y[idx]

        if self.transform:
            X = self.transform(X)
        return X, y

class MLP(nn.Module):
    def __init__(self, num_in, num_hidden, num_out):
        super().__init__()
        self.flatten = nn.Flatten(1, -1)
        self.l1 = nn.Linear(num_in, num_hidden) # 隠れ層(第1層)を定義
        self.l2 = nn.Linear(num_hidden, num_out) # 隠れ層(第2層)を定義

    def forward(self, x):
        x = self.flatten(x)
        # z1 = self.l1(x) 
        # a1 = F.relu(z1)
        # z2 = self.l2(a1)
        x = self.l2(F.relu(self.l1(x)))
        return x

# データ準備
dataset = datasets.load_digits()
data = dataset['data']
target = dataset['target']
images = dataset['images']
images = images * (255. / 16.) # 0~16 -> 0~255
images = images.astype(np.uint8)
X_train, X_val, y_train, y_val = train_test_split(images, target, test_size = 0.2, random_state = 0)

transform = transforms.Compose([transforms.ToTensor(), 
                                transforms.Normalize((0.5,), (0.5,))])
train_mydataset = MyDataset(X_train, y_train, transform = transform)
val_mydataset = MyDataset(X_val, y_val, transform = transform)

train_myloader = DataLoader(train_mydataset, batch_size = 10, shuffle = True, num_workers = 2)
val_myloader = DataLoader(val_mydataset, batch_size = 10, num_workers = 2)


learning_rate = 0.03
train_losses = []
val_losses = []
val_accuracies = []
# モデルの初期化
model = MLP(28*28, 30, 10)
# optimizerの定義
opt = optim.SGD(model.parameters(), lr = learning_rate)

# モデル学習
for epoch in range(5):
    running_loss = 0
    running_val_loss = 0
    running_val_accuracy = 0
    
    for train_batch, data in enumerate(train_loader):

        X, y = data
        opt.zero_grad() # 勾配初期化
        # forward
        preds = model(X)
        loss = F.cross_entropy(preds, y)
        running_loss += loss.item()

        # backward
        loss.backward()
        opt.step() # パラメータ更新
        

    with torch.no_grad():
        for val_batch, data in enumerate(val_loader):
            X_val, y_val = data
            preds_val = model(X_val)
            val_loss = F.cross_entropy(preds_val, y_val)
            running_val_loss += val_loss.item()
            val_accuracy = torch.sum(torch.argmax(preds_val, dim = -1) == y_val) / y_val.shape[0]
            running_val_accuracy += val_accuracy.item()

    train_losses.append(running_loss / (train_batch + 1))
    val_losses.append(running_val_loss / (val_batch + 1))
    val_accuracies.append(running_val_accuracy / (val_batch + 1))
    print(f'epoch:{epoch}, train error:{train_losses[-1]}, val_losses:{val_losses[-1]}, val_accuracy:{val_accuracies[-1]}')